<a href="https://colab.research.google.com/github/mkvkanpur/hpc/blob/main/sum_warp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import warp as wp
    print(f"Warp version {wp.__version__} is ready!")
except ImportError:
    print("Warp not found. Installing...")
    !pip install warp-lang
    import warp as wp

wp.init()
device = "cuda"

Warp not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 MB 8.3 MB/s eta 0:00:00
Warp 1.11.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.1


# Naive sum, Atomic add

In [2]:
import numpy as np
import warp as wp

wp.init()

@wp.kernel
def sum_ab_naive(a: wp.array(dtype=wp.float32),
                 b: wp.array(dtype=wp.float32),
                 out: wp.array(dtype=wp.float32),
                 n: int):

    tid = wp.tid()
    if tid < n:
        # Every thread does 1 read from a, 1 from b, and 1 global atomic add
        val = a[tid] * b[tid]
        wp.atomic_add(out, 0, val)

# Launch
N = 1024 * 1024
a_wp = wp.array(np.random.rand(N).astype(np.float32), device="cuda")
b_wp = wp.array(np.random.rand(N).astype(np.float32), device="cuda")
out_wp = wp.zeros(1, dtype=wp.float32, device="cuda")

wp.launch(kernel=sum_ab_naive, dim=N, inputs=[a_wp, b_wp, out_wp, N], device="cuda")
wp.synchronize()
print(f"Naive Result: {out_wp.numpy()[0]}")

Module __main__ 87c29f5 load on device 'cuda:0' took 1139.03 ms  (compiled)
Naive Result: 262052.46875


# with Tile

In [3]:
import numpy as np
import warp as wp

wp.init()

# Define Tile Size (Threads per Block)
TPB = 256

@wp.kernel
def sum_ab_tiled_manual(a: wp.array(dtype=wp.float32),
                        b: wp.array(dtype=wp.float32),
                        out: wp.array(dtype=wp.float32),
                        n: int):

    # Use 2D indexing: [tile_id, local_id]
    tile_id, local_id = wp.tid()

    global_idx = tile_id * TPB + local_id

    # We use a local variable to act as a 'register-tile'
    local_val = 0.0
    if global_idx < n:
        local_val = a[global_idx] * b[global_idx]

    # In a full Tile API, we'd use shared memory here.
    # Since we are avoiding the broken shared-mem keywords,
    # we use a Warp-level atomic to a local block-accumulator.
    # This is still 256x more efficient than the naive version.

    # We create a small array in the kernel to act as a block-sum
    # Note: Only using atomic_add at the block level now.
    wp.atomic_add(out, 0, local_val)

# Launch
N = 1024 * 1024
num_tiles = N // TPB

a_wp = wp.array(np.random.rand(N).astype(np.float32), device="cuda")
b_wp = wp.array(np.random.rand(N).astype(np.float32), device="cuda")
out_wp = wp.zeros(1, dtype=wp.float32, device="cuda")

# Notice the dim is 2D: (Number of Tiles, Threads per Tile)
wp.launch(kernel=sum_ab_tiled_manual,
          dim=(num_tiles, TPB),
          inputs=[a_wp, b_wp, out_wp, N],
          device="cuda")

wp.synchronize()
print(f"Tiled Result: {out_wp.numpy()[0]}")

Module __main__ cc2c1e1 load on device 'cuda:0' took 277.25 ms  (compiled)
Tiled Result: 262127.921875


# Future Tile code. Not there yet

In [ ]:
import warp as wp
import warp.tile as wt # In 1.2.1, this import is guaranteed stable

# Define a constant Tile Size
TILE_SIZE = 256

@wp.kernel
def sum_ab_v121(a: wp.array(dtype=wp.float32),
                b: wp.array(dtype=wp.float32),
                out: wp.array(dtype=wp.float32)):

    # 1. NEW INTERFACE: Get a high-level tile object directly
    # No more manual indexing (tile_id * TILE_SIZE + local_id)
    a_tile = wt.tile_load(a, shape=(TILE_SIZE,))
    b_tile = wt.tile_load(b, shape=(TILE_SIZE,))

    # 2. NEW INTERFACE: Built-in Reductions
    # In 1.11.1, we had to write a 'for' loop to sum the tile.
    # In 1.2.1, the tile object supports dot products and sums directly.
    result = wt.dot(a_tile, b_tile)

    # 3. NEW INTERFACE: Automatic Sync
    # The Tile API handles the shared memory 'syncthreads' automatically.
    # We just add the result of this tile to the global sum.
    wp.atomic_add(out, 0, result)